# Chapter 10 — EDA와 시각화로 공격 패턴 발견 (실습)

## 실습 목표
1. 라벨별 분포 비교 (히스토그램·박스플롯)
2. 상관관계 heatmap
3. 시간대 line chart
4. pairplot

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib
# !pip install -q seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
rng = np.random.default_rng(42)

# ch09 의 데이터 셋업 재사용
def make_rows(n, label, dur, pkts, syn, ports):
    return pd.DataFrame({
        'Destination Port': rng.choice(ports, size=n),
        'Flow Duration': rng.uniform(*dur, size=n),
        'Total Fwd Packets': rng.integers(*pkts, size=n),
        'Total Backward Packets': rng.integers(0, 10, size=n) if label != 'BENIGN' else rng.integers(*pkts, size=n),
        'SYN Flag Count': rng.uniform(*syn, size=n),
        'ACK Flag Count': rng.uniform(0.0 if label != 'BENIGN' else 0.4, 0.6, size=n),
        'Timestamp': pd.to_datetime('2024-07-05') + pd.to_timedelta(rng.integers(0, 86400, size=n), unit='s'),
        'Label': label,
    })
df = pd.concat([
    make_rows(16000, 'BENIGN',   (1000, 1_000_000), (5, 100),    (0, 0.05),  [80, 443, 53, 22]),
    make_rows( 2300, 'DDoS',     (10, 5000),         (500, 5000), (0.4, 0.7), [80, 443]),
    make_rows( 1200, 'PortScan', (1, 200),           (1, 5),      (0.85, 1.0), list(range(1, 1024))),
], ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)
COLORS = {'BENIGN': '#1a5f7a', 'DDoS': '#e74c3c', 'PortScan': '#c0392b'}

---
## Step 1. 라벨별 박스플롯

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 라벨별 박스플롯
# 만들 결과: palette
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
fig, axes = ...
for ax, feat in zip(axes, ['Flow Duration', 'Total Fwd Packets', 'SYN Flag Count']):
    sns.boxplot(data = ...
                palette = ...
    if feat != 'SYN Flag Count':
        ...
```
</details>


---
## Step 2. 상관관계 heatmap

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 상관관계 heatmap
# 만들 결과: num_cols, corr
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
num_cols = ...
corr = ...
fig, ax = ...
sns.heatmap(corr, annot = ...
...
```
</details>


---
## Step 3. 시간대 라인 차트

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 시간대 라인 차트
# 만들 결과: df['hour'], hourly
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
df['hour'] = ...
hourly = ...
fig, ax = ...
for label, color in COLORS.items():
    if label in hourly.columns:
        ax.plot(hourly.index, hourly[label], marker = ...
...
```
</details>


---
## Step 4. pairplot

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. pairplot
# 만들 결과: sample, hue
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
sample = ...
...
             hue = ...
...
```
</details>


---
## 정리

- 박스플롯·상관관계·시간 차트·pairplot 으로 라벨별 패턴이 한눈에 보인다.
- 이 정보로 다음 챕터(ch11)의 ML 모델이 어떤 feature 를 활용할지 미리 예측 가능.